# 01 — Compute Gaze Angle

**Purpose:** Combine IMU `pitch` with gaze `elevation` to produce a `gaze angle` time series.

## Pipeline

1. Set the `DATA_DIR` to the folder containing your Pupil Neon export (must have `imu.csv` and `gaze_positions.csv`).
2. Load and synchronize the two data streams (gaze is downsampled to the IMU frame rate).
3. Compute `gaze angle = pitch + elevation`.
4. Visualize the result and optionally save to `gaze_angle.csv`.

## Outputs

- `gaze_angle.csv` — columns: `timestamp [ns]`, `gaze angle [deg]`, `pitch [deg]`, `elevation [deg]`, `time_sec`

In [ ]:
import sys, os

# Add the project root to the path so we can import neon_gaze
sys.path.insert(0, os.path.abspath(".."))

from neon_gaze.io import load_imu, load_gaze_positions, save_gaze_angle
from neon_gaze.processing import synchronize_gaze_to_imu, compute_gaze_angle
from neon_gaze.gait import detect_gait_events
from neon_gaze.plotting import (
    plot_gaze_angle,
    plot_gaze_angle_with_gait,
    plot_gaze_histogram,
    plot_imu_yaw,
    plot_filtered_acceleration,
)

## Configuration

Point `DATA_DIR` at a Pupil Neon export folder. The demo data shipped with this repo lives at `../demo/input/`.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
# Point this at any Pupil Neon export folder containing imu.csv and
# gaze_positions.csv.  The demo data uses a small sample.
DATA_DIR = "../demo/input"

# Where to save the output gaze_angle.csv
OUTPUT_DIR = "../demo/output"

# Set to True to write the output CSV when running this notebook
SAVE_OUTPUT = False

# Label for plot titles (e.g. condition name)
CONDITION_LABEL = "demo"

## Step 1 — Load raw data

In [ ]:
imu_df = load_imu(os.path.join(DATA_DIR, "imu.csv"))
gaze_df = load_gaze_positions(os.path.join(DATA_DIR, "gaze_positions.csv"))

print("IMU data:")
display(imu_df.head())
print("\nGaze positions data:")
display(gaze_df.head())

## Step 2 — Synchronize and compute gaze angle

In [ ]:
merged_df = synchronize_gaze_to_imu(imu_df, gaze_df)
gaze_angle_df = compute_gaze_angle(merged_df)

print(f"Gaze angle dataframe: {len(gaze_angle_df)} rows")
display(gaze_angle_df.head())

## Step 3 — Visualize

In [ ]:
fig = plot_imu_yaw(imu_df, title=f"IMU Yaw Over Time — {CONDITION_LABEL}")
fig.show()

In [ ]:
fig = plot_gaze_angle(gaze_angle_df, title=f"Gaze Angle Over Time — {CONDITION_LABEL}")
fig.show()

In [ ]:
fig = plot_gaze_histogram(gaze_angle_df, title=f"Histogram of Gaze Angle — {CONDITION_LABEL}")
fig.show()

## Step 4 — Gait event detection (optional)

Detect heel-strike and toe-off events from the IMU vertical acceleration, and overlay them on the gaze-angle plot.

In [ ]:
events = detect_gait_events(imu_df)

print(f"Heel-strikes detected: {len(events.hs_times)}")
print(f"Toe-offs detected:     {len(events.to_times)}")

In [ ]:
fig = plot_filtered_acceleration(
    events.imu_df, events.hs_times, events.to_times,
    title=f"Filtered IMU Vertical Acceleration — {CONDITION_LABEL}",
)
fig.show()

In [ ]:
fig = plot_gaze_angle_with_gait(
    gaze_angle_df, events.hs_times, events.to_times,
    title=f"Gaze Angle with Gait Events — {CONDITION_LABEL}",
)
fig.show()

## Step 5 — Save output

In [ ]:
if SAVE_OUTPUT:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    save_gaze_angle(gaze_angle_df, os.path.join(OUTPUT_DIR, "gaze_angle.csv"))
else:
    print("SAVE_OUTPUT is False — set to True in the config cell to save.")

## Interactive metric explorer

Use the dropdown to toggle between `elevation` and `pitch` over time.

In [ ]:
from ipywidgets import widgets
import plotly.graph_objects as go

def plot_gaze_metric(metric):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=gaze_angle_df["time_sec"],
        y=gaze_angle_df[metric],
        mode="lines+markers",
        name=metric,
        line=dict(width=2),
        marker=dict(size=4),
    ))
    fig.update_layout(
        title=f"{metric} Over Time",
        xaxis_title="Time (sec)",
        yaxis_title=metric,
        yaxis=dict(range=[-90, 45]),
    )
    fig.show()

dropdown = widgets.Dropdown(
    options=["elevation [deg]", "pitch [deg]"],
    value="elevation [deg]",
    description="Metric:",
)
widgets.interact(plot_gaze_metric, metric=dropdown)